In [ ]:
!pip -q install earthengine-api geopandas rasterio shapely fiona

In [ ]:
import os
import time
import glob
import shutil
import ee
import rasterio
import geopandas as gpd

from datetime import datetime, timedelta, timezone

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

# ------------------------------------------------------------
# GOOGLE EARTH ENGINE PROJECT ("ee-username")
# ------------------------------------------------------------
gee_project_id = None

# ------------------------------------------------------------
# DEFINE A ROOT FOR THE PROJECT
# ------------------------------------------------------------
PROJECT_ROOT = "/content/drive/MyDrive/U-Net_burned_areas_project"

# ------------------------------------------------------------
# SATELLITE SELECTION:  "sentinel" | "landsat" | "both"
# ------------------------------------------------------------
satellite_choice = "landsat"

# ------------------------------------------------------------
# PRE-FIRE AND POST-FIRE DATES  (YYYY-MM-DD)
# ------------------------------------------------------------
pre_date  = "YYYY-MM-DD"
post_date = "YYYY-MM-DD"

# ------------------------------------------------------------
# MAXIMUM DATE RELAXATION
# ------------------------------------------------------------
max_day_relaxation = 5

# ------------------------------------------------------------
# AOI POLYGON
# ------------------------------------------------------------
aoi_polygons_file = 'XXX_polygons.gpkg'

# ------------------------------------------------------------
# DOWNLOAD PURPOSE
# ------------------------------------------------------------
# "training"   → saves to data/sites/<site_name>/<Satellite>/
# "prediction" → saves to Pre+Post_Downloads/<site_name>/
download_purpose = "prediction"   # <-- change to "training" if needed

# ------------------------------------------------------------
# SITE NAME  (used in BOTH training and prediction)
# ------------------------------------------------------------
site_name = "site-XXX-tropical-savanna"   # <-- change to your site name

# ------------------------------------------------------------
# SENTINEL-2 BAND SELECTION
# ------------------------------------------------------------
# Available bands in S2_SR_HARMONIZED (Level-2A).
# B10 (Cirrus) does NOT exist in Level-2A — do not add it.
# Remove any band name from both lists to exclude it from the download.
# The two lists must stay the same length.
sentinel_bands = [
    "B1",  "B2",    "B3",    "B4",  "B5",        "B6",        "B7",
    "B8",  "B8A",   "B9",    "B11", "B12"
]
sentinel_band_names = [
    "COASTAL", "BLUE", "GREEN", "RED", "RED_EDGE1", "RED_EDGE2", "RED_EDGE3",
    "NIR",     "NIR_NARROW", "WATER_VAPOUR", "SWIR1", "SWIR2"
]

# ------------------------------------------------------------
# LANDSAT 8/9 BAND SELECTION
# ------------------------------------------------------------
# Available SR bands in LANDSAT/LC08-09/C02/T1_L2.
# Remove any band name from both lists to exclude it from the download.
# The two lists must stay the same length.
landsat_bands = [
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"
]
landsat_band_names = [
    "COASTAL", "BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"
]

# ============================================================
# INPUT CLEANING AND CHECKS
# ============================================================
satellite_choice = satellite_choice.lower().strip()
download_purpose = download_purpose.lower().strip()

if satellite_choice not in ["sentinel", "landsat", "both"]:
    raise ValueError("satellite_choice must be: 'sentinel', 'landsat', or 'both'")

if download_purpose not in ["training", "prediction"]:
    raise ValueError("download_purpose must be: 'training' or 'prediction'")

if pre_date == "YYYY-MM-DD" or post_date == "YYYY-MM-DD":
    raise ValueError("Please replace pre_date and post_date with real dates.")

if len(sentinel_bands) != len(sentinel_band_names):
    raise ValueError("sentinel_bands and sentinel_band_names must have the same length.")

if len(landsat_bands) != len(landsat_band_names):
    raise ValueError("landsat_bands and landsat_band_names must have the same length.")

if download_purpose == "prediction":
    output_folder = f"{PROJECT_ROOT}/Prediction_Input/Input_Rasters_For_Prediction/{site_name}"
else:
    output_folder = f"{PROJECT_ROOT}/data/sites/{site_name}"

os.makedirs(output_folder, exist_ok=True)

print("Inputs loaded successfully.")
print("GEE project ID  :", gee_project_id)
print("Satellite        :", satellite_choice)
print("Pre-fire date   :", pre_date)
print("Post-fire date  :", post_date)
print("AOI polygon     :", aoi_polygons_file)
print("Download purpose:", download_purpose)
print("Site name        :", site_name)
print("Base output dir :", output_folder)
print(f"Sentinel bands  : {sentinel_bands}")
print(f"Landsat bands   : {landsat_bands}")

Inputs loaded successfully.
GEE project ID  : None
Satellite        : landsat
Pre-fire date   : 2019-12-12
Post-fire date  : 2020-01-12
AOI polygon     : XXX_polygons.gpkg
Download purpose: prediction
Site name        : site-XXX-tropical-savanna
Base output dir : /content/drive/MyDrive/NewStructure/User's_Inputs/Input_Rasters_For_Prediction/site-XXX-tropical-savanna
Sentinel bands  : ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
Landsat bands   : ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']


In [ ]:
if download_purpose == "prediction":
    polygon_path    = f"{PROJECT_ROOT}/Prediction_Input/AOI_Polygons/{aoi_polygons_file}"
elif download_purpose == 'training':
    polygon_path    = f"{PROJECT_ROOT}/data/sites/{site_name}/{aoi_polygons_file}"
gee_temp_folder = "GEE_TEMP_EXPORTS"

print("Polygon path    :", polygon_path)
print("GEE temp folder :", gee_temp_folder)

Polygon path    : /content/drive/MyDrive/NewStructure/data/sites/site-413-tropical-savanna/413_polygons.gpkg
GEE temp folder : GEE_TEMP_EXPORTS


In [ ]:
ee.Authenticate()

if gee_project_id is None:
    ee.Initialize()
else:
    ee.Initialize(project=gee_project_id)

print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


In [ ]:
# ============================================================
# LOAD AOI FROM USER POLYGON
# ============================================================

if not os.path.exists(polygon_path):
    raise FileNotFoundError(f"Polygon file not found: {polygon_path}")

gdf = gpd.read_file(polygon_path)

if gdf.empty:
    raise ValueError("The polygon file is empty.")

gdf = gdf.to_crs(epsg=4326)

minx, miny, maxx, maxy = gdf.total_bounds

buffer_deg = 0.01

aoi = ee.Geometry.Rectangle([
    float(minx - buffer_deg),
    float(miny - buffer_deg),
    float(maxx + buffer_deg),
    float(maxy + buffer_deg)
])

print("AOI loaded successfully.")
print("AOI bounds:", minx, miny, maxx, maxy)

AOI loaded successfully.
AOI bounds: -11.877154161977284 13.309374528687096 -11.639072385377693 13.513808389571382


In [ ]:
# ============================================================
# PREPROCESSING FUNCTIONS
# ============================================================
# Band selections are controlled by sentinel_bands / sentinel_band_names
# and landsat_bands / landsat_band_names defined in the config cell above.

def prepare_sentinel(img):
    img = img.select(sentinel_bands, sentinel_band_names)
    img = img.divide(10000)
    return img.copyProperties(img, img.propertyNames())


def prepare_landsat(img):
    optical = img.select(landsat_bands, landsat_band_names)
    optical = optical.multiply(0.0000275).add(-0.2)
    optical = optical.clamp(0, 1)
    return optical.copyProperties(img, img.propertyNames())

In [ ]:
# ============================================================
# COLLECTION FUNCTIONS
# ============================================================

def get_sentinel_collection_raw(aoi, start, end):
    """Raw collection — no map applied, all properties preserved."""
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
    )

def get_landsat_collection_raw(aoi, start, end):
    """Raw collection — no map applied, all properties preserved."""
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    return (
        l8.merge(l9)
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUD_COVER", 80))
    )

def get_sentinel_collection(aoi, start, end):
    """Preprocessed collection — for final export only."""
    return get_sentinel_collection_raw(aoi, start, end).map(prepare_sentinel)

def get_landsat_collection(aoi, start, end):
    """Preprocessed collection — for final export only."""
    return get_landsat_collection_raw(aoi, start, end).map(prepare_landsat)

In [ ]:
import os
import glob
import time
import shutil
from datetime import datetime, timedelta, timezone

gee_temp_folder = "GEE_TEMP_EXPORTS"


def check_aoi_coverage(collection, aoi, threshold=0.98):
    """
    Checks whether the selected image collection covers the AOI.
    """

    mosaic_geom = collection.geometry()

    aoi_area = aoi.area(maxError=1)
    intersection_area = mosaic_geom.intersection(aoi, maxError=1).area(maxError=1)

    coverage_fraction = intersection_area.divide(aoi_area).getInfo()

    is_covered = coverage_fraction >= threshold

    return coverage_fraction, is_covered


def harmonize_and_mosaic(collection, aoi, cloud_property):
    """
    Sorts images by cloud cover, mosaics them, clips to AOI,
    and keeps the standard bands prepared in previous cells.
    """

    img = collection.sort(cloud_property).mosaic().clip(aoi)

    return img


def find_best_image(sensor, target_date):
    target_dt = datetime.strptime(target_date, "%Y-%m-%d")

    for delta in range(1, max_day_relaxation + 1):

        start_dt = target_dt - timedelta(days=delta)
        end_dt   = target_dt + timedelta(days=delta + 1)

        start = start_dt.strftime("%Y-%m-%d")
        end   = end_dt.strftime("%Y-%m-%d")

        print(f"\nSearching {sensor.upper()} within ±{delta} days")
        print("Date range:", start, "to", end)

        if sensor == "sentinel":
            raw_col = get_sentinel_collection_raw(aoi, start, end)
            cloud_property = "CLOUDY_PIXEL_PERCENTAGE"
        elif sensor == "landsat":
            raw_col = get_landsat_collection_raw(aoi, start, end)
            cloud_property = "CLOUD_COVER"
        else:
            raise ValueError("sensor must be 'sentinel' or 'landsat'")

        count = raw_col.size().getInfo()
        print("Images found:", count)

        if count == 0:
            continue

        # ── Check if the FULL window mosaic covers the AOI ───────────
        if sensor == "sentinel":
            full_col = get_sentinel_collection(aoi, start, end)
        else:
            full_col = get_landsat_collection(aoi, start, end)

        coverage_fraction, is_covered = check_aoi_coverage(full_col, aoi)
        print("AOI coverage fraction (full window mosaic):", coverage_fraction)

        if not is_covered:
            print("AOI is not fully covered even with full window. Trying wider date range...")
            continue

        # ── Coverage OK: now pick the best single date if possible ────────
        ids    = raw_col.aggregate_array("system:index").getInfo()
        clouds = raw_col.aggregate_array(cloud_property).getInfo()
        times  = raw_col.aggregate_array("system:time_start").getInfo()

        candidates = []
        for i in range(len(ids)):
            img_dt   = datetime.fromtimestamp(times[i] / 1000, tz=timezone.utc)
            date_str = img_dt.strftime("%Y-%m-%d")
            diff_days = abs((img_dt.date() - target_dt.date()).days)
            candidates.append({
                "id":    ids[i],
                "date":  date_str,
                "diff":  diff_days,
                "cloud": clouds[i] if clouds[i] is not None else 999
            })

        candidates = sorted(candidates, key=lambda x: (x["diff"], x["cloud"]))

        # Try each candidate date from best to worst
        for cand in candidates:
            best_date  = cand["date"]
            best_start = best_date
            best_end   = (datetime.strptime(best_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")

            if sensor == "sentinel":
                best_col = get_sentinel_collection(aoi, best_start, best_end)
            else:
                best_col = get_landsat_collection(aoi, best_start, best_end)

            single_coverage, single_ok = check_aoi_coverage(best_col, aoi)
            print(f"  Trying {best_date}: coverage={single_coverage:.3f}, cloud={cand['cloud']:.1f}")

            if single_ok:
                print(f"Selected single date: {best_date}")
                best_img = harmonize_and_mosaic(best_col, aoi, cloud_property)
                return best_img, best_date

        # ── No single date covers AOI ─ use the full-window mosaic ────────
        print(f"No single date fully covers AOI. Using multi-date mosaic over ±{delta} days.")
        best_img = harmonize_and_mosaic(full_col, aoi, cloud_property)
        return best_img, f"{start}_to_{end}"

    raise Exception(
        f"No suitable {sensor} image found within ±{max_day_relaxation} days."
    )


def export_and_move(img, description, filename, scale, sensor, final_output_dir):
    temp_drive_path = f"/content/drive/MyDrive/{gee_temp_folder}"

    os.makedirs(temp_drive_path, exist_ok=True)
    os.makedirs(final_output_dir, exist_ok=True)

    # Delete old temp files with same prefix
    for old_file in glob.glob(temp_drive_path + f"/{filename}*.tif"):
        os.remove(old_file)

    # Delete old final output with same filename
    for old_file in glob.glob(final_output_dir + f"/{filename}*.tif"):
        os.remove(old_file)

    print("Final destination:", final_output_dir)

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=description,
        folder=gee_temp_folder,
        fileNamePrefix=filename,
        region=aoi,
        scale=scale,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )

    task.start()

    print("Export task started:", description)

    while True:
        status = task.status()
        state = status["state"]

        if state in ["READY", "RUNNING"]:
            print(f"  [{state}] waiting …")
            time.sleep(20)
            continue

        if state == "COMPLETED":
            print(f"  [{state}] export completed.")
            break

        if state in ["FAILED", "CANCELLED"]:
            raise RuntimeError(f"Export failed/cancelled: {status}")

    print("Waiting for exported file to appear in Colab Drive mount...")

    tif_files = []

    for attempt in range(60):
        tif_files = glob.glob(temp_drive_path + f"/{filename}*.tif")

        if tif_files:
            break

        print(f"File not visible yet... waiting 10 seconds ({attempt+1}/60)")
        time.sleep(10)

    if not tif_files:
        raise FileNotFoundError(
            f"No .tif files found in {temp_drive_path} for prefix '{filename}'. "
            "The export finished, but Google Drive has not synced to Colab yet."
        )

    src = tif_files[0]
    dst = os.path.join(final_output_dir, os.path.basename(src))

    if os.path.exists(dst):
        os.remove(dst)

    shutil.move(src, dst)

    print("Moved to:", dst)

In [ ]:
# ============================================================
# RUN DOWNLOADS (PRE-FIRE)
# ============================================================

if satellite_choice in ["sentinel", "both"]:

    print("\n==============================")
    print("DOWNLOADING SENTINEL PRE-FIRE IMAGE")
    print("==============================")

    sentinel_output_dir = os.path.join(output_folder, "Sentinel")
    os.makedirs(sentinel_output_dir, exist_ok=True)

    pre_img, selected_pre_date = find_best_image("sentinel", pre_date)

    export_and_move(
        img=pre_img,
        description="Sentinel_PRE",
        filename="Sentinel_PRE",
        scale=10,
        sensor="sentinel",
        final_output_dir=sentinel_output_dir
    )

    print(f"\nSentinel PRE date used : {selected_pre_date}")


if satellite_choice in ["landsat", "both"]:

    print("\n==============================")
    print("DOWNLOADING LANDSAT PRE-FIRE IMAGE")
    print("==============================")

    landsat_output_dir = os.path.join(output_folder, "Landsat")
    os.makedirs(landsat_output_dir, exist_ok=True)

    pre_img, selected_pre_date = find_best_image("landsat", pre_date)

    export_and_move(
        img=pre_img,
        description="Landsat_PRE",
        filename="Landsat_PRE",
        scale=30,
        sensor="landsat",
        final_output_dir=landsat_output_dir
    )

    print(f"\nLandsat PRE date used : {selected_pre_date}")


DOWNLOADING LANDSAT PRE-FIRE IMAGE

Searching LANDSAT within ±1 days
Date range: 2023-11-06 to 2023-11-09
Images found: 0

Searching LANDSAT within ±2 days
Date range: 2023-11-05 to 2023-11-10
Images found: 2
AOI coverage fraction (full window mosaic): 1
  Trying 2023-11-09: coverage=1.000, cloud=2.9
Selected single date: 2023-11-09
Final destination: /content/drive/MyDrive/NewStructure/data/sites/site-413-tropical-savanna/Landsat
Export task started: Landsat_PRE
  [READY] waiting …
  [RUNNING] waiting …
  [RUNNING] waiting …
  [COMPLETED] export completed.
Waiting for exported file to appear in Colab Drive mount...
Moved to: /content/drive/MyDrive/NewStructure/data/sites/site-413-tropical-savanna/Landsat/Landsat_PRE.tif

Landsat PRE date used : 2023-11-09


In [ ]:
# ============================================================
# RUN DOWNLOADS (POST-FIRE)
# ============================================================

if satellite_choice in ["sentinel", "both"]:

    print("\n==============================")
    print("DOWNLOADING SENTINEL POST-FIRE IMAGE")
    print("==============================")

    # Ensure directory exists, in case previous cell was skipped or modified
    sentinel_output_dir = os.path.join(output_folder, "Sentinel")
    os.makedirs(sentinel_output_dir, exist_ok=True)

    post_img, selected_post_date = find_best_image("sentinel", post_date)

    export_and_move(
        img=post_img,
        description="Sentinel_POST",
        filename="Sentinel_POST",
        scale=10,
        sensor="sentinel",
        final_output_dir=sentinel_output_dir
    )

    print(f"\nSentinel POST date used: {selected_post_date}")


if satellite_choice in ["landsat", "both"]:

    print("\n==============================")
    print("DOWNLOADING LANDSAT POST-FIRE IMAGE")
    print("==============================")

    # Ensure directory exists, in case previous cell was skipped or modified
    landsat_output_dir = os.path.join(output_folder, "Landsat")
    os.makedirs(landsat_output_dir, exist_ok=True)

    post_img, selected_post_date = find_best_image("landsat", post_date)

    export_and_move(
        img=post_img,
        description="Landsat_POST",
        filename="Landsat_POST",
        scale=30,
        sensor="landsat",
        final_output_dir=landsat_output_dir
    )

    print(f"\nLandsat POST date used: {selected_post_date}")


print("\nALL DOWNLOADS FINISHED.")
print(f"Files saved to their respective subfolders within: {output_folder}")


DOWNLOADING LANDSAT POST-FIRE IMAGE

Searching LANDSAT within ±1 days
Date range: 2023-11-26 to 2023-11-29
Images found: 1
AOI coverage fraction (full window mosaic): 0.0006617969707883552
AOI is not fully covered even with full window. Trying wider date range...

Searching LANDSAT within ±2 days
Date range: 2023-11-25 to 2023-11-30
Images found: 3
AOI coverage fraction (full window mosaic): 1
  Trying 2023-11-26: coverage=0.001, cloud=8.4
  Trying 2023-11-25: coverage=1.000, cloud=0.2
Selected single date: 2023-11-25
Final destination: /content/drive/MyDrive/NewStructure/data/sites/site-413-tropical-savanna/Landsat
Export task started: Landsat_POST
  [READY] waiting …
  [RUNNING] waiting …
  [RUNNING] waiting …
  [RUNNING] waiting …
  [COMPLETED] export completed.
Waiting for exported file to appear in Colab Drive mount...
File not visible yet... waiting 10 seconds (1/60)
File not visible yet... waiting 10 seconds (2/60)
File not visible yet... waiting 10 seconds (3/60)
File not visi

## Check number of saved bands


In [ ]:
import rasterio
import os

def check_image_channels(directory):
    print(f"\nChecking images in: {directory}")
    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return

    found_images = False
    for filename in os.listdir(directory):
        if filename.endswith('.tif'):
            filepath = os.path.join(directory, filename)
            try:
                with rasterio.open(filepath) as src:
                    num_channels = src.count
                    print(f"  {filename}: {num_channels} channels")
                found_images = True
            except rasterio.errors.RasterioIOError as e:
                print(f"  Error opening {filename}: {e}")

    if not found_images:
        print(f"  No GeoTIFF images found in {directory}")

# Check Sentinel images
if 'sentinel_output_dir' in locals() and os.path.exists(sentinel_output_dir):
    check_image_channels(sentinel_output_dir)
else:
    # Fallback in case sentinel_output_dir was not explicitly set or path invalid
    print("Sentinel output directory variable not found or path invalid. Trying to reconstruct.")
    sentinel_output_dir_fallback = os.path.join(output_folder, "Sentinel")
    check_image_channels(sentinel_output_dir_fallback)

# Check Landsat images (if Landsat was chosen or could be chosen in the future)
if 'landsat_output_dir' in locals() and os.path.exists(landsat_output_dir):
    check_image_channels(landsat_output_dir)
elif satellite_choice in ["landsat", "both"]:
    print("Landsat output directory variable not found or path invalid. Trying to reconstruct.")
    landsat_output_dir_fallback = os.path.join(output_folder, "Landsat")
    check_image_channels(landsat_output_dir_fallback)
else:
    print("Landsat images were not selected for download.")



Checking images in: /content/drive/MyDrive/Geoinformatics_Project/NewStructure/data/sites/site-413-tropical-savanna/Sentinel
  Sentinel_PRE.tif: 12 channels
  Sentinel_POST.tif: 12 channels
Landsat images were not selected for download.


##### Contributors:
- **Ayman Mutasim Alfadul Abdelgadir**: aymanmutasim@mail.polimi.it
- **Evgenii Miasnikov**: evgenii.miasnikov@mail.polimi.it

```
   Copyright 2026 Ayman Mutasim Alfadul Abdelgadir, Evgenii Miasnikov

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       https://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```